In [ ]:
import os
import random
import math
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# ============================================================
# 1. Settings
# ============================================================

OUTPUT_FOLDER = "png_grafer"
CSV_FILE = os.path.join(OUTPUT_FOLDER, "correct_graph_answers.csv")

N_MIN = 1
N_MAX = 24
GRAPHS_PER_NODE_COUNT = 10

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ============================================================
# 2. Generate connected graph
# ============================================================

def generate_random_graph(n_nodes):
    """
    Generates an undirected connected graph.
    """

    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))

    if n_nodes == 1:
        return G

    # Make graph connected with a random spanning tree
    for node in range(1, n_nodes):
        previous_node = random.randint(0, node - 1)
        G.add_edge(node, previous_node)

    # Add extra random edges
    possible_edges = [
        (i, j)
        for i in range(n_nodes)
        for j in range(i + 1, n_nodes)
        if not G.has_edge(i, j)
    ]

    max_extra_edges = min(len(possible_edges), n_nodes)
    n_extra_edges = random.randint(0, max_extra_edges)

    if n_extra_edges > 0:
        extra_edges = random.sample(possible_edges, n_extra_edges)
        G.add_edges_from(extra_edges)

    return G

# ============================================================
# 3. Assign random red/blue colors
# ============================================================

def assign_node_colors(G):
    return {
        node: random.choice(["red", "blue"])
        for node in G.nodes()
    }

# ============================================================
# 4. Layout helper to reduce node overlap
# ============================================================

def min_node_distance(pos):
    positions = list(pos.values())

    if len(positions) <= 1:
        return float("inf")

    min_dist = float("inf")

    for i in range(len(positions)):
        for j in range(i + 1, len(positions)):
            x1, y1 = positions[i]
            x2, y2 = positions[j]

            dist = math.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)
            min_dist = min(min_dist, dist)

    return min_dist


def get_good_layout(G, graph_seed):
    """
    Tries several layouts and keeps the one where nodes are best separated.
    """

    n_nodes = G.number_of_nodes()

    if n_nodes == 1:
        return {0: (0, 0)}

    if n_nodes <= 8:
        return nx.circular_layout(G, scale=2.8)

    best_pos = None
    best_distance = -1

    for attempt in range(25):
        pos = nx.spring_layout(
            G,
            seed=graph_seed + attempt,
            k=3.0 / math.sqrt(n_nodes),
            iterations=600,
            scale=3.5
        )

        distance = min_node_distance(pos)

        if distance > best_distance:
            best_distance = distance
            best_pos = pos

    return best_pos

# ============================================================
# 5. Generate graphs and save PNG + CSV rows
# ============================================================

answer_rows = []
graph_id_counter = 1

for n_nodes in range(N_MIN, N_MAX + 1):

    for graph_number in range(1, GRAPHS_PER_NODE_COUNT + 1):

        G = generate_random_graph(n_nodes)

        node_colors = assign_node_colors(G)
        colors_for_plot = [node_colors[node] for node in G.nodes()]

        n_edges = G.number_of_edges()
        n_red_nodes = sum(1 for color in node_colors.values() if color == "red")
        n_blue_nodes = sum(1 for color in node_colors.values() if color == "blue")

        image_id = f"graph_{graph_id_counter:03d}_n{n_nodes:02d}"
        filename = f"{image_id}.png"
        filepath = os.path.join(OUTPUT_FOLDER, filename)

        pos = get_good_layout(G, RANDOM_SEED + graph_id_counter)

        # Dynamic size settings
        figsize = 5.5 if n_nodes <= 8 else 6.5 if n_nodes <= 16 else 7.5
        node_size = 700 if n_nodes <= 8 else 450 if n_nodes <= 16 else 300

        plt.figure(figsize=(figsize, figsize))

        nx.draw_networkx_edges(
            G,
            pos,
            edge_color="black",
            width=1.5,
            alpha=0.75
        )

        nx.draw_networkx_nodes(
            G,
            pos,
            node_color=colors_for_plot,
            node_size=node_size,
            edgecolors="black",
            linewidths=1.4
        )

        # Labels are disabled so the model cannot read node numbers
        # nx.draw_networkx_labels(G, pos, font_size=8, font_color="white")

        plt.axis("off")
        plt.margins(0.20)
        plt.tight_layout()

        plt.savefig(
            filepath,
            format="png",
            dpi=300,
            bbox_inches="tight",
            pad_inches=0.25
        )

        plt.close()

        # Correct answers saved for evaluation
        answer_rows.append({
            "image_id": image_id,
            "filename": filename,
            "filepath": filepath,
            "n_nodes_true": n_nodes,
            "n_edges_true": n_edges,
            "n_red_nodes_true": n_red_nodes,
            "n_blue_nodes_true": n_blue_nodes,
            "directed_true": False,
            "graph_number_for_node_count": graph_number,
            "node_level": n_nodes,
            "random_seed": RANDOM_SEED + graph_id_counter
        })

        graph_id_counter += 1

# ============================================================
# 6. Save CSV file with correct answers
# ============================================================

answers_df = pd.DataFrame(answer_rows)
answers_df.to_csv(CSV_FILE, index=False)

# ============================================================
# 7. Done
# ============================================================

print("Done.")
print(f"{len(answers_df)} PNG-filer er gemt i mappen: {OUTPUT_FOLDER}")
print(f"CSV-fil med korrekte svar er gemt som: {CSV_FILE}")

print("\nFørste 5 rækker i CSV-filen:")
print(answers_df.head())

print("\nSidste 5 rækker i CSV-filen:")
print(answers_df.tail())